# Pregunta Inicial

## 1. ¿Qué factores explican la Puntuación promedio que recibe un anime en MyAnimeList?

## Carga de Datos para el metodo de Regresión Lineal

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# ==============================
# 1. Carga del conjunto de datos
# ==============================
df = pd.read_csv("data.csv", encoding="latin1")

# ==============================
# 2. Selección de variables relevantes
# ==============================

# Variables cuantitativas
vars_cuantitativas = [
    'Episodes',
    'Duration_Minutes',
    'Score',
    'Scored_Users',
    'Ranked',
    'Popularity',
    'Members'
]

# Variables categóricas relevantes
vars_categoricas = [
    'Type',
    'Source',
    'Rating'
]

df_selected = df[vars_cuantitativas + vars_categoricas]

# ==============================
# 3. Conversión a tipos numéricos
# ==============================
df_selected = df[vars_cuantitativas + vars_categoricas].copy()

df_selected.loc[:, vars_cuantitativas] = df_selected[vars_cuantitativas].apply(
    pd.to_numeric, errors='coerce'
)


# ==============================
# 4. Manejo de valores faltantes
# ==============================

# Eliminación de filas con valores nulos
df_clean = df_selected.dropna()

# ==============================
# 5. Codificación de variables categóricas
# ==============================

df_encoded = pd.get_dummies(
    df_clean,
    columns=vars_categoricas,
    drop_first=True
)

# ==============================
# 6. Transformación logarítmica
# ==============================

vars_log = ['Episodes', 'Scored_Users', 'Members']

for col in vars_log:
    df_encoded[f'log_{col}'] = np.log1p(df_encoded[col])

# ==============================
# 7. Estandarización de variables numéricas
# ==============================

vars_escalar = [
    'log_Episodes',
    'Duration_Minutes',
    'Score',
    'log_Scored_Users',
    'Ranked',
    'Popularity',
    'log_Members'
]

scaler = StandardScaler()
df_encoded[vars_escalar] = scaler.fit_transform(df_encoded[vars_escalar])

# ==============================
# 8. Dataset final listo para análisis
# ==============================

df_final = df_encoded.copy()


# ==============================
# 9. Definición de X e Y
# ==============================

# Variable dependiente (forzar a float)
y = df_final['Score'].astype(float)

# Variables independientes
X = df_final.drop(columns=['Score'])

# ==============================
# 10. Corrección DEFINITIVA de tipos
# ==============================

# Convertir TODAS las columnas a float
X = X.astype(float)

# ==============================
# 11. Agregar constante
# ==============================

X = sm.add_constant(X)

# ==============================
# 12. Alineación y limpieza final
# ==============================

valid_idx = X.dropna().index
X = X.loc[valid_idx]
y = y.loc[valid_idx]

# ==============================
# 13. Ajuste del modelo
# ==============================

model = sm.OLS(y, X)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                  Score   R-squared:                       0.961
Model:                            OLS   Adj. R-squared:                  0.961
Method:                 Least Squares   F-statistic:                     9008.
Date:                Thu, 01 Jan 2026   Prob (F-statistic):               0.00
Time:                        09:07:43   Log-Likelihood:                 2576.8
No. Observations:               12937   AIC:                            -5082.
Df Residuals:                   12901   BIC:                            -4813.
Df Model:                          35                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

## Resumen de los resultados del modelo inicial

El modelo de regresión lineal múltiple estimado para explicar la puntuación promedio (Score) de los animes en MyAnimeList presenta un alto poder explicativo, con un R² ajustado de 0.961, lo que indica que el conjunto de variables incluidas explica aproximadamente el 96 % de la variabilidad de la puntuación promedio.

Entre las variables cuantitativas, se observa que la duración por episodio tiene un efecto positivo y estadísticamente significativo sobre la puntuación, lo que sugiere que animes con episodios más largos tienden a recibir mejores valoraciones. Asimismo, las variables asociadas a popularidad e interacción de usuarios (como el número de usuarios que puntuaron y el número de miembros) muestran efectos significativos, aunque con signos opuestos, reflejando la compleja relación entre alcance masivo y valoración promedio.

La variable Ranked destaca por presentar un coeficiente negativo de gran magnitud y una significancia estadística extremadamente alta. Este resultado indica una relación casi perfecta entre el ranking del anime y su puntuación promedio, lo cual es consistente con la forma en que MyAnimeList construye el ranking a partir del propio Score.

En cuanto a las variables categóricas, el tipo de anime resulta relevante, observándose diferencias significativas entre formatos como TV, OVA, Music y Special respecto a la categoría base. De igual forma, algunas fuentes de origen (como Original o Light Novel) y ciertas clasificaciones por edad (Rating) presentan efectos estadísticamente significativos sobre la puntuación promedio.

A pesar del excelente ajuste global, el modelo presenta problemas metodológicos importantes que afectan la interpretabilidad de los coeficientes:

Endogeneidad :

La variable Ranked está directamente construida a partir de la puntuación promedio. Su inclusión introduce una relación circular, inflando artificialmente el R² y dominando el modelo.

Multicolinealidad severa :

El elevado condition number (2.55e+07) indica una fuerte correlación entre varias variables explicativas, especialmente aquellas relacionadas con popularidad: Ranked , Popularity , Scored_Users / log_Scored_Users , Members / log_Members

Estas variables capturan dimensiones muy similares del mismo fenómeno, lo que genera inestabilidad en los coeficientes y dificulta la interpretación individual de los efectos.

## Cambios propuestos para el siguiente script

Con el fin de obtener un modelo más parsimonioso, estable e interpretable, se realizarán los siguientes ajustes en la siguiente iteración del análisis:

- Eliminación de la variable Ranked :
Se eliminará esta variable por razones de endogeneidad, dado que no constituye un factor explicativo independiente de la puntuación promedio.

- Reducción de variables redundantes de popularidad :
Se mantendrán únicamente las versiones logarítmicas más representativas de la popularidad (por ejemplo, log_Members y log_Scored_Users), eliminando sus versiones originales (Members, Scored_Users) y evitando la duplicación de información.

- Reestimación del modelo : Con el conjunto reducido de variables, se reestimará el modelo de regresión lineal múltiple para: Reducir la multicolinealidad , Obtener coeficientes más estables , Lograr un R² más realista , Facilitar la interpretación de los efectos individuales .


Estos ajustes permiten alinear el modelo con buenas prácticas estadísticas, priorizando la interpretabilidad sobre el ajuste artificialmente elevado. El modelo refinado permitirá responder de forma más precisa la pregunta de investigación, identificando qué características estructurales, de formato y de popularidad influyen realmente en la puntuación promedio de los animes en MyAnimeList.

In [3]:
import statsmodels.api as sm

# ==============================
# 1. Variable dependiente
# ==============================

y = df_final['Score'].astype(float)

# ==============================
# 2. Variables independientes (modelo refinado)
# ==============================

X = df_final.drop(columns=[
    'Score',          # variable dependiente
    'Ranked',         # endogeneidad
    'Members',        # redundante con log_Members
    'Scored_Users'    # redundante con log_Scored_Users
])

# ==============================
# 3. Conversión explícita a float
# ==============================

X = X.astype(float)

# ==============================
# 4. Agregar constante
# ==============================

X = sm.add_constant(X)

# ==============================
# 5. Alineación y limpieza final
# ==============================

valid_idx = X.dropna().index
X = X.loc[valid_idx]
y = y.loc[valid_idx]

# ==============================
# 6. Ajuste del modelo refinado
# ==============================

model_refined = sm.OLS(y, X)
results_refined = model_refined.fit()

print(results_refined.summary())


                            OLS Regression Results                            
Dep. Variable:                  Score   R-squared:                       0.593
Model:                            OLS   Adj. R-squared:                  0.592
Method:                 Least Squares   F-statistic:                     588.7
Date:                Thu, 01 Jan 2026   Prob (F-statistic):               0.00
Time:                        09:56:29   Log-Likelihood:                -12534.
No. Observations:               12937   AIC:                         2.513e+04
Df Residuals:                   12904   BIC:                         2.538e+04
Df Model:                          32                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

## Resultados del modelo de regresión lineal múltiple (modelo refinado)